# Phase 3: DCA-Trie v2 — Dynamic Trie Expansion During Beam Search

**Goal:** Demonstrate step-wise dynamic KG-Trie expansion at reasoning time.
V2 expands the pruned trie (V1) on-the-fly when the model commits to an entity,
adding semantically relevant 1-hop neighbors.

**Key Idea:**
- V1 (Phase 2) statically prunes the KG-Trie by ~50% at tau=0.45 (FNR=1%).
- V2 re-expands around committed entities during beam search, recovering
  relevant paths without re-admitting all irrelevant paths.

**Requires:** A100 GPU (40GB VRAM).

**Pipeline:**
1. Load model, tokenizer, data
2. Build GCR baseline trie, prune it, expand dynamically
3. SIR comparison: baseline vs pruned vs expanded
4. Qualitative examples of expansion
5. (Optional) End-to-end V2 generation on 1 question

## 0. Pin NumPy < 2 (required by GCR's sklearn dependency)

Colab ships NumPy 2.x which breaks sklearn's internal imports (`numpy.char` removed).
This cell pins NumPy, then restarts the kernel so the change takes effect.

In [ ]:
import os
os.system("pip install -q 'numpy<2'")
os.kill(os.getpid(), 9)  # restart runtime

## 1. Environment Setup

After runtime restart, clone repo, init submodule, and install dca-trie.

In [1]:
import sys, os, json, warnings, gc, copy
import numpy as np
import torch
from pprint import pprint

IN_COLAB = 'google.colab' in sys.modules
print(f"Python: {sys.version}")
print(f"Running in Colab: {IN_COLAB}")
print(f"NumPy: {np.__version__}")

cuda_ok = torch.cuda.is_available()
if cuda_ok:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {vram_gb:.1f} GB")
    if vram_gb < 20:
        print("WARNING: <20GB VRAM. Use 4-bit quantization or fewer beams.")
else:
    print("WARNING: No GPU detected. 8B model requires a GPU.")

if IN_COLAB:
    ROOT = "/content/dca-trie"
    if not os.path.exists(ROOT):
        !git clone https://github.com/adjanour/dca-trie.git {ROOT}
        %cd {ROOT}
        !git submodule update --init
    else:
        %cd {ROOT}
        !git pull --rebase
    !pip install -q -e .
else:
    print("Running locally. Ensure poetry env is active.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Running in Colab: True
NumPy: 1.26.4
GPU: NVIDIA A100-SXM4-40GB  VRAM: 42.4 GB
/content/dca-trie
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 790 bytes | 790.00 KiB/s, done.
From https://github.com/adjanour/dca-trie
   b1edda1..479cd60  master     -> origin/master
Updating b1edda1..479cd60
Fast-forward
 phase3_dynamic_trie_decoding.ipynb | 29 +++++++++++++++--------------
 1 file changed, 15 insertions(+), 14 deletions(-)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dca-trie (pyproject.toml) ... done


In [2]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF_TOKEN configured.")
else:
    raise ValueError("HF_TOKEN required. Set it above or in .env")

HF_TOKEN configured.


## 2. Load Model, Tokenizer, and Data

GCR-Meta-Llama-3.1-8B-Instruct in bf16 with SDPA (~20GB VRAM).

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig
from gcr.src.qa_prompt_builder import GraphConstrainedPromptBuilder
from gcr.src.trie import MarisaTrie, Trie
from gcr.src.utils.graph_utils import build_graph, dfs, path_to_string
from gcr.src.utils import load_data, get_truth_paths

MODEL_NAME = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, padding_side="left"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map="auto",
)
model.eval()
print(f"Model loaded on {model.device}")

data_path = "rmanluo"
dataset_name = "RoG-webqsp"
dataset = load_data(data_path, dataset_name, split="test")
print(f"Dataset: {len(dataset)} questions")

ImportError: cannot import name 'path_to_string' from 'gcr.src.utils.graph_utils' (/content/dca-trie/gcr/src/utils/graph_utils.py)

## 3. Scorer, MID Resolver, SIR Measurer, V1/V2 Builders

In [ ]:
from dca_trie.semantic_scorer import SemanticScorer
from dca_trie.mid_resolver import MidResolver
from dca_trie.sir_measurement import SIRMeasurer
from dca_trie.v1_trie_builder import V1TrieBuilder
from dca_trie.v2_decoder import V2Decoder, BeamTrieState, TrieStateManager

scorer = SemanticScorer()
print(f"Scorer device: {scorer.device}")

resolver = MidResolver()
resolver.build_from_dataset(dataset)
cov = resolver.coverage(dataset)
print(f"MID coverage: {cov['coverage_pct']}% ({cov['resolved']}/{cov['total_mids']})")

sir_measurer = SIRMeasurer(scorer, tokenizer)


def resolved_path_to_str(path):
    return resolver.resolve_path(path_to_string(path))


def marisa_to_mutable(marisa_trie):
    trie = Trie()
    for seq in marisa_trie:
        trie.add(seq)
    return trie


TAU = 0.45  # from threshold sweep: 1% FNR, 50% reduction on 100 questions

## 4. Helper Functions

In [ ]:
def build_gcr_trie(q_dict, index_path_length=2):
    g = build_graph(q_dict["graph"])
    all_paths = dfs(g, q_dict["q_entity"], index_path_length)
    if not all_paths:
        return None, 0
    path_strs = [resolved_path_to_str(p) for p in all_paths]
    tokenized = tokenizer(path_strs, padding=False, add_special_tokens=False).input_ids
    tokenized = [ids + [tokenizer.eos_token_id] for ids in tokenized]
    trie = MarisaTrie(tokenized, max_token_id=len(tokenizer) + 1)
    return trie, len(path_strs)


def measure_sir(trie, question_text):
    result = sir_measurer.measure_from_trie(trie, question_text)
    return result["sir"], result["num_paths"]


def v2_expand_from_entity(trie, entity, q_dict, g, tau):
    if not g.has_node(entity):
        return 0
    query_emb = scorer.encode_query(q_dict["question"])
    added = 0
    for neighbor in g.successors(entity):
        rel = g[entity][neighbor].get("relation", "")
        path_str = resolver.resolve_path(f"{entity} -> {rel} -> {neighbor}")
        score = scorer.score_path(path_str, query_emb)
        if score >= tau:
            seq = tokenizer(path_str, add_special_tokens=False).input_ids
            try:
                trie.add(seq + [tokenizer.eos_token_id])
                added += 1
            except Exception:
                pass
    return added


def demo_question(q_dict):
    print(f"ID: {q_dict.get('id', '?')}")
    print(f"Question: {q_dict['question']}")
    print(f"Q Entity: {resolver.resolve(q_dict['q_entity'])}")
    gold = q_dict.get("ground_truth_paths", [])
    print(f"Gold paths ({len(gold)}):")
    for p in gold:
        print(f"  {resolver.resolve_path(p)}")

## 5. Single Question Deep Dive

Show all three trie stages for the first question.

In [ ]:
q = dataset[0]
demo_question(q)

v1_builder = V1TrieBuilder(tokenizer, scorer, tau=TAU,
                           path_to_str_fn=resolved_path_to_str)

orig_trie, orig_size = build_gcr_trie(q)
orig_sir, _ = measure_sir(orig_trie, q["question"])
print(f"\n--- Stage 1: Original GCR Trie ---")
print(f"  Size: {orig_size} paths")
print(f"  SIR:  {orig_sir:.4f}")

pruned_marisa = v1_builder.build_filtered_trie(q)
if pruned_marisa:
    pruned_sir, pruned_size = measure_sir(pruned_marisa, q["question"])
    reduction = 1 - pruned_size / orig_size if orig_size > 0 else 0
else:
    pruned_sir, pruned_size, reduction = None, 0, 1.0
print(f"\n--- Stage 2: V1 Pruned (tau={TAU}) ---")
print(f"  Size: {pruned_size} paths (reduction: {reduction:.1%})")
print(f"  SIR:  {pruned_sir:.4f}" if pruned_sir is not None else "  SIR:  N/A (empty)")

g = build_graph(q["graph"])
v2_trie = marisa_to_mutable(pruned_marisa)
total_added = 0
entities_expanded = set()
for gp_path in q.get("ground_truth_paths", []):
    parts = gp_path.split(" -> ")
    for i in range(0, len(parts), 2):
        entity = parts[i]
        if entity not in entities_expanded:
            added = v2_expand_from_entity(v2_trie, entity, q, g, TAU)
            total_added += added
            if added > 0:
                print(f"  From '{resolver.resolve(entity)}': +{added} paths")
            entities_expanded.add(entity)

v2_sir, v2_paths = measure_sir(v2_trie, q["question"])
print(f"\n--- Stage 3: V2 Expanded ---")
print(f"  Entities expanded: {len(entities_expanded)}")
print(f"  Paths: {pruned_size} -> {v2_paths} (+{v2_paths - pruned_size})")
print(f"  SIR:  {orig_sir:.4f} -> {pruned_sir:.4f} -> {v2_sir:.4f}")
print(f"  SIR change from pruned: {v2_sir - pruned_sir:+.4f}")

## 6. Multi-Question SIR Analysis

Compare SIR across all three trie stages for N questions.

In [ ]:
def analyze_question(q_dict):
    res = {"id": q_dict.get("id", "?"), "question": q_dict["question"]}

    orig_trie, orig_size = build_gcr_trie(q_dict)
    if orig_trie is None:
        return res | {"orig_size": 0, "orig_sir": None, "error": "no paths"}
    orig_sir, _ = measure_sir(orig_trie, q_dict["question"])
    res["orig_size"] = orig_size
    res["orig_sir"] = round(orig_sir, 4) if orig_sir is not None else None

    pruned_marisa = v1_builder.build_filtered_trie(q_dict)
    if pruned_marisa is None:
        res.update({"pruned_size": 0, "pruned_sir": None,
                     "v2_size": 0, "v2_sir": None, "reduction": 1.0})
        return res

    pruned_sir, pruned_size = measure_sir(pruned_marisa, q_dict["question"])
    res["pruned_size"] = pruned_size
    res["pruned_sir"] = round(pruned_sir, 4) if pruned_sir is not None else None
    res["reduction"] = round(1 - pruned_size / orig_size, 4)

    v2_trie = marisa_to_mutable(pruned_marisa)
    g = build_graph(q_dict["graph"])
    expanded_entities = 0
    for gp_path in q_dict.get("ground_truth_paths", []):
        parts = gp_path.split(" -> ")
        for i in range(0, len(parts), 2):
            entity = parts[i]
            added = v2_expand_from_entity(v2_trie, entity, q_dict, g, TAU)
            if added > 0:
                expanded_entities += 1

    v2_sir, v2_paths = measure_sir(v2_trie, q_dict["question"])
    res["v2_size"] = v2_paths
    res["v2_sir"] = round(v2_sir, 4) if v2_sir is not None else None
    res["v2_expansions"] = expanded_entities
    res["paths_recovered"] = v2_paths - pruned_size
    return res


NUM_QUESTIONS = 20
print(f"Analyzing {NUM_QUESTIONS} questions at tau={TAU}...")
results = []
for i, q_dict in enumerate(dataset[:NUM_QUESTIONS]):
    r = analyze_question(q_dict)
    results.append(r)
    lane = f"orig={r.get('orig_sir')} pruned={r.get('pruned_sir')} v2={r.get('v2_sir')}"
    sizes = f"{r.get('orig_size')}->{r.get('pruned_size')}->{r.get('v2_size')}"
    print(f"  [{i+1}/{NUM_QUESTIONS}] {r.get('id')}: {lane} | {sizes}")

In [ ]:
print("\n" + "=" * 80)
print("PHASE 3: DCA-Trie v2 SIR Results")
print("=" * 80)

valid = [r for r in results if r.get("orig_sir") is not None]
print(f"\nQuestions: {len(valid)}")

o_sirs = [r["orig_sir"] for r in valid if r["orig_sir"] is not None]
p_sirs = [r["pruned_sir"] for r in valid if r["pruned_sir"] is not None]
v_sirs = [r["v2_sir"] for r in valid if r["v2_sir"] is not None]

print(f"\n  {'Stage':<25} {'Mean SIR':<12} {'Std SIR':<12} {'Avg Paths':<12}")
print(f"  {'-'*61}")
hdr = lambda name, sirs, sizes: f"  {name:<25} {np.mean(sirs):<12.4f} {np.std(sirs):<12.4f} {np.mean(sizes):<12.1f}"
print(hdr("Original GCR", o_sirs, [r["orig_size"] for r in valid]))
print(hdr("V1 Pruned (tau=0.45)", p_sirs, [r["pruned_size"] for r in valid]))
if v_sirs:
    print(hdr("V2 Expanded", v_sirs, [r["v2_size"] for r in valid]))

avg_red = np.mean([r.get("reduction", 0) for r in valid])
avg_rec = np.mean([r.get("paths_recovered", 0) for r in valid])
avg_exp = np.mean([r.get("v2_expansions", 0) for r in valid])
print(f"\n--- Expansion Stats ---")
print(f"  Avg trie reduction (V1 vs baseline): {avg_red:.1%}")
print(f"  Avg paths recovered by V2:           {avg_rec:.1f}")
print(f"  Avg entity expansions per question:  {avg_exp:.1f}")

## 7. Qualitative Expansion Examples

Show which 1-hop paths V2 adds (or leaves pruned) from gold-path entities.

In [ ]:
def show_expansion(q_dict, top_k=6):
    g = build_graph(q_dict["graph"])
    query_emb = scorer.encode_query(q_dict["question"])
    seen = set()
    print(f"\nQuestion: {q_dict['question']}")
    for gp in q_dict.get("ground_truth_paths", []):
        parts = gp.split(" -> ")
        for i in range(0, len(parts), 2):
            e = parts[i]
            if e in seen or not g.has_node(e):
                continue
            seen.add(e)
            neighbors = list(g.successors(e))[:top_k]
            print(f"\n  From: {resolver.resolve(e)}")
            for nb in neighbors:
                rel = g[e][nb].get("relation", "")
                ps = resolver.resolve_path(f"{e} -> {rel} -> {nb}")
                sc = scorer.score_path(ps, query_emb)
                tag = "ADDED" if sc >= TAU else "pruned"
                print(f"    {ps:<70} {sc:.3f} [{tag}]")


for q_dict in dataset[:3]:
    show_expansion(q_dict)

## 8. (Optional) V2Decoder.generate() — Full Beam Search

Runs the actual HuggingFace generation with dynamic trie expansion.
Takes ~3-5 min per question on A100.

**Skip this cell for a quick demo.** It is here for end-to-end verification.

In [ ]:
%%time

def run_v2_generation(q_dict):
    prompt_builder = GraphConstrainedPromptBuilder(
        tokenizer=tokenizer, prompt="zero-shot",
        undirected=False, index_path_length=2,
    )
    input_text, ground_paths, _ = prompt_builder.process_input(q_dict)

    pruned_marisa = v1_builder.build_filtered_trie(q_dict)
    if pruned_marisa is None:
        return {"outputs": [], "error": "empty pruned trie"}
    initial_trie = marisa_to_mutable(pruned_marisa)

    start_ids = tokenizer.encode("# Reasoning Path:\n", add_special_tokens=False)
    end_ids   = tokenizer.encode("\n", add_special_tokens=False)

    decoder = V2Decoder(
        model=model, tokenizer=tokenizer, scorer=scorer,
        tau=TAU, k=5, path_to_str_fn=resolved_path_to_str, max_hops=2,
    )

    outputs = decoder.generate(
        input_query=input_text,
        initial_trie=initial_trie,
        question_dict=q_dict,
        start_token_ids=start_ids,
        end_token_ids=end_ids,
        num_beams=5,
        max_new_tokens=128,
    )
    return {"outputs": outputs, "ground_truth": ground_paths}


print("V2Decoder on 1 question with beams=5...")
q_dict = dataset[0]
res = run_v2_generation(q_dict)
print(f"Outputs: {len(res['outputs'])} beams")
if res['outputs']:
    print(f"\nTop beam:\n{res['outputs'][0]}")
    print(f"\nGold paths:")
    for gp in q_dict.get("ground_truth_paths", []):
        print(f"  {resolver.resolve_path(gp)}")

## 9. Conclusion

In [ ]:
output = {
    "tau": TAU,
    "num_questions": len(valid),
    "results": valid,
    "summary": {
        "mean_orig_sir": float(np.mean(o_sirs)),
        "mean_pruned_sir": float(np.mean(p_sirs)),
        "mean_v2_sir": float(np.mean(v_sirs)) if v_sirs else None,
        "mean_reduction": float(avg_red),
        "mean_recovery": float(avg_rec),
    }
}
with open("data/phase3_v2_results.json", "w") as f:
    json.dump(output, f, indent=2)
print(f"Saved results to data/phase3_v2_results.json")
print(f"\n--- Quick Reference ---")
print(f"Baseline SIR: {np.mean(o_sirs):.4f}")
print(f"V1 Pruned SIR: {np.mean(p_sirs):.4f} (reduction: {avg_red:.1%})")
if v_sirs:
    print(f"V2 Expanded SIR: {np.mean(v_sirs):.4f} (paths recovered: {avg_rec:.1f}/q)")